In [ ]:
pip install escnn

# Filter Heatmaps

Author: Robin

This notebook visualizes learned first-layer filters for the three CIFAR-10 models stored alongside this experiment folder:

- `cnngeneral.py`
- `gcnngeneral.py`
- `gcnn_p4mgeneral.py`

For portability, keep the notebook, these copied model-definition files, and the `.pth` checkpoints in the same folder whenever possible.

## Portable Folder Layout

Recommended contents of this `Filter Visualization` folder:

- `filter_heatmaps.ipynb`
- `cnngeneral.py`
- `gcnngeneral.py`
- `gcnn_p4mgeneral.py`
- `cnngeneral.pth`
- `gcnngeneral.pth`
- `gcnn_p4mgeneral.pth`

If you only want the plain CNN heatmaps, you only need `cnngeneral.py` and `cnngeneral.pth` in this folder.

## Colab Notes

For Google Colab, upload this notebook together with whichever copied model files and checkpoints you want to use from this folder.

You do not need the whole repository if these copied files are present here.

In [1]:
from pathlib import Path
import sys
import math
import subprocess

import torch
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import google.colab
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

print(f'Running in Colab: {IN_COLAB}')

Running in Colab: False


In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'seaborn'], check=True)
    print('Installed seaborn for this Colab session.')
else:
    print('Skipping Colab-only install cell.')

In [ ]:
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    print(f'Uploaded {len(uploaded)} file(s).')
else:
    print('Skipping upload cell because this is not Colab.')

In [ ]:
def find_experiment_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'filter_heatmaps.ipynb').exists() and (candidate / 'cnngeneral.py').exists():
            return candidate
    return start

EXPERIMENT_DIR = find_experiment_dir(Path.cwd().resolve())

if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

print(f'Notebook working directory: {Path.cwd().resolve()}')
print(f'Experiment directory: {EXPERIMENT_DIR}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
def import_cnn_model():
    from cnngeneral import Net
    return Net


def import_gcnn_p4_model():
    from gcnngeneral import GCNN
    return GCNN


def import_gcnn_p4m_model():
    from gcnn_p4mgeneral import GCNNP4M
    return GCNNP4M

In [ ]:
CHECKPOINT_CANDIDATES = {
    'cnn': [
        EXPERIMENT_DIR / 'cnngeneral.pth',
        EXPERIMENT_DIR / 'models' / 'trained_cnn' / 'cnngeneral.pth',
    ],
    'gcnn_p4': [
        EXPERIMENT_DIR / 'gcnngeneral.pth',
        EXPERIMENT_DIR / 'models' / 'trained_cnn' / 'gcnngeneral.pth',
        EXPERIMENT_DIR / 'models' / 'trainned_gcnn' / 'gcnngeneral.pth',
    ],
    'gcnn_p4m': [
        EXPERIMENT_DIR / 'gcnn_p4mgeneral.pth',
        EXPERIMENT_DIR / 'models' / 'trained_cnn' / 'gcnn_p4mgeneral.pth',
        EXPERIMENT_DIR / 'models' / 'trainned_gcnn' / 'gcnn_p4mgeneral.pth',
    ],
}

DISPLAY_NAMES = {
    'cnn': 'CNN (Z2)',
    'gcnn_p4': 'GCNN (p4)',
    'gcnn_p4m': 'GCNN (p4m)',
}

In [ ]:
def resolve_checkpoint(model_key):
    for path in CHECKPOINT_CANDIDATES[model_key]:
        if path.exists():
            return path
    raise FileNotFoundError(
        f'No checkpoint found for {model_key}. Checked: ' + ', '.join(str(p) for p in CHECKPOINT_CANDIDATES[model_key])
    )


def build_model(model_key):
    if model_key == 'cnn':
        model = import_cnn_model()()
    elif model_key == 'gcnn_p4':
        model = import_gcnn_p4_model()()
    elif model_key == 'gcnn_p4m':
        model = import_gcnn_p4m_model()()
    else:
        raise ValueError(f'Unknown model key: {model_key}')

    checkpoint_path = resolve_checkpoint(model_key)
    state_dict = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(state_dict)
    model.eval()
    return model, checkpoint_path


def first_standard_conv(model):
    for module in model.modules():
        if isinstance(module, torch.nn.Conv2d):
            return module
    raise ValueError('No torch.nn.Conv2d layer found.')


def first_group_conv(model):
    for module in model.modules():
        if module.__class__.__name__ == 'R2Conv':
            return module
    raise ValueError('No escnn.nn.R2Conv layer found.')


def extract_first_layer_filters(model, model_key):
    if model_key == 'cnn':
        weights = first_standard_conv(model).weight.detach().cpu()
    else:
        group_conv = first_group_conv(model)
        if hasattr(group_conv, 'expand_parameters'):
            weights, _ = group_conv.expand_parameters()
        elif hasattr(group_conv, 'filter'):
            weights = group_conv.filter
        else:
            raise ValueError('Could not extract expanded filters from the group convolution layer.')
        weights = weights.detach().cpu()
    return weights


def to_spatial_heatmaps(weights, mode='mean_abs'):
    if weights.ndim != 4:
        raise ValueError(f'Expected 4D filter tensor, got shape {tuple(weights.shape)}')

    if mode == 'mean_abs':
        heatmaps = weights.abs().mean(dim=1)
    elif mode == 'mean':
        heatmaps = weights.mean(dim=1)
    else:
        raise ValueError("mode must be 'mean_abs' or 'mean'")

    return heatmaps


def plot_filter_grid(heatmaps, title, max_filters=36, cmap='mako'):
    count = min(max_filters, heatmaps.shape[0])
    cols = 6
    rows = math.ceil(count / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(2.4 * cols, 2.2 * rows))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    vmin = float(heatmaps[:count].min())
    vmax = float(heatmaps[:count].max())

    for idx in range(len(axes)):
        ax = axes[idx]
        if idx < count:
            sns.heatmap(
                heatmaps[idx],
                ax=ax,
                cmap=cmap,
                cbar=False,
                square=True,
                xticklabels=False,
                yticklabels=False,
                vmin=vmin,
                vmax=vmax,
            )
            ax.set_title(f'Filter {idx}', fontsize=9)
        else:
            ax.axis('off')

    fig.suptitle(title, fontsize=16)
    fig.tight_layout()
    plt.show()


def summarize_filter_tensor(weights, model_name):
    print(model_name)
    print(f'Filter tensor shape: {tuple(weights.shape)}')
    print(f'Min value: {weights.min().item():.4f}')
    print(f'Max value: {weights.max().item():.4f}')
    print(f'Mean absolute value: {weights.abs().mean().item():.4f}')

## Choose A Model

Set `MODEL_KEY` to one of:

- `'cnn'`
- `'gcnn_p4'`
- `'gcnn_p4m'`

In [ ]:
MODEL_KEY = 'cnn'

model, checkpoint_path = build_model(MODEL_KEY)
weights = extract_first_layer_filters(model, MODEL_KEY)
heatmaps = to_spatial_heatmaps(weights, mode='mean_abs')

print(f'Loaded checkpoint: {checkpoint_path}')
summarize_filter_tensor(weights, DISPLAY_NAMES[MODEL_KEY])
plot_filter_grid(heatmaps, f"{DISPLAY_NAMES[MODEL_KEY]} First-Layer Filter Heatmaps")

## Compare All Available Models

This cell tries to load every model checkpoint it can find and plots the first few filters for each.

In [ ]:
for model_key in ['cnn', 'gcnn_p4', 'gcnn_p4m']:
    try:
        model, checkpoint_path = build_model(model_key)
        weights = extract_first_layer_filters(model, model_key)
        heatmaps = to_spatial_heatmaps(weights, mode='mean_abs')
        print(f'Loaded {DISPLAY_NAMES[model_key]} from {checkpoint_path}')
        summarize_filter_tensor(weights, DISPLAY_NAMES[model_key])
        plot_filter_grid(heatmaps, f"{DISPLAY_NAMES[model_key]} First-Layer Filter Heatmaps", max_filters=24)
    except FileNotFoundError as exc:
        print(exc)
    except ModuleNotFoundError as exc:
        print(f'Missing dependency for {model_key}: {exc}')
    except Exception as exc:
        print(f'Could not visualize {model_key}: {exc}')